This notebook illustrates the agent creation process for the **LLM 20 Questions**. Running this notebook produces a `submission.tar.gz` file. You may submit this file directly from the **Submit to competition** heading to the right. Alternatively, from the notebook viewer, click the *Output* tab then find and download `submission.tar.gz`. Click **Submit Agent** at the upper-left of the competition homepage to upload your file and make your submission. 

In [1]:
%%bash
cd /kaggle/working
pip install -q -U -t /kaggle/working/submission/lib immutabledict sentencepiece
git clone https://github.com/google/gemma_pytorch.git > /dev/null
mkdir /kaggle/working/submission/lib/gemma/
mv /kaggle/working/gemma_pytorch/gemma/* /kaggle/working/submission/lib/gemma/

Cloning into 'gemma_pytorch'...


In [2]:
%%writefile submission/main.py
# Setup
import os
import sys

# **IMPORTANT:** Set up your system path like this to make your code work
# both in notebooks and in the simulations environment.
KAGGLE_AGENT_PATH = "/kaggle_simulations/agent/"
if os.path.exists(KAGGLE_AGENT_PATH):
    sys.path.insert(0, os.path.join(KAGGLE_AGENT_PATH, 'lib'))
else:
    sys.path.insert(0, "/kaggle/working/submission/lib")

import contextlib
import os
import sys
from pathlib import Path

import torch
from gemma.config import get_config_for_7b, get_config_for_2b
from gemma.model import GemmaForCausalLM

if os.path.exists(KAGGLE_AGENT_PATH):
    WEIGHTS_PATH = os.path.join(KAGGLE_AGENT_PATH, "gemma/pytorch/7b-it-quant/2")
else:
    WEIGHTS_PATH = "/kaggle/input/gemma/pytorch/7b-it-quant/2"

# Prompt Formatting
import itertools
from typing import Iterable


class GemmaFormatter:
    _start_token = '<start_of_turn>'
    _end_token = '<end_of_turn>'

    def __init__(self, system_prompt: str = None, few_shot_examples: Iterable = None):
        self._system_prompt = system_prompt
        self._few_shot_examples = few_shot_examples
        self._turn_user = f"{self._start_token}user\n{{}}{self._end_token}\n"
        self._turn_model = f"{self._start_token}model\n{{}}{self._end_token}\n"
        self.reset()

    def __repr__(self):
        return self._state

    def user(self, prompt):
        self._state += self._turn_user.format(prompt)
        return self

    def model(self, prompt):
        self._state += self._turn_model.format(prompt)
        return self

    def start_user_turn(self):
        self._state += f"{self._start_token}user\n"
        return self

    def start_model_turn(self):
        self._state += f"{self._start_token}model\n"
        return self

    def end_turn(self):
        self._state += f"{self._end_token}\n"
        return self

    def reset(self):
        self._state = ""
        if self._system_prompt is not None:
            self.user(self._system_prompt)
        if self._few_shot_examples is not None:
            self.apply_turns(self._few_shot_examples, start_agent='user')
        return self

    def apply_turns(self, turns: Iterable, start_agent: str):
        formatters = [self.model, self.user] if start_agent == 'model' else [self.user, self.model]
        formatters = itertools.cycle(formatters)
        for fmt, turn in zip(formatters, turns):
            fmt(turn)
        return self


# Agent Definitions
import re


@contextlib.contextmanager
def _set_default_tensor_type(dtype: torch.dtype):
    """Set the default torch dtype to the given dtype."""
    torch.set_default_dtype(dtype)
    yield
    torch.set_default_dtype(torch.float)


class GemmaAgent:
    def __init__(self, variant='7b-it-quant', device='cuda:0', system_prompt=None, few_shot_examples=None):
        self._variant = variant
        self._device = torch.device(device)
        self.formatter = GemmaFormatter(system_prompt=system_prompt, few_shot_examples=few_shot_examples)

        print("Initializing model")
        model_config = get_config_for_2b() if "2b" in variant else get_config_for_7b()
        model_config.tokenizer = os.path.join(WEIGHTS_PATH, "tokenizer.model")
        model_config.quant = "quant" in variant

        with _set_default_tensor_type(model_config.get_dtype()):
            model = GemmaForCausalLM(model_config)
            ckpt_path = os.path.join(WEIGHTS_PATH , f'gemma-{variant}.ckpt')
            model.load_weights(ckpt_path)
            self.model = model.to(self._device).eval()

    def __call__(self, obs, *args):
        self._start_session(obs)
        prompt = str(self.formatter)
        response = self._call_llm(prompt)
        response = self._parse_response(response, obs)
        print(f"{response=}")
        return response

    def _start_session(self, obs: dict):
        raise NotImplementedError

    def _call_llm(self, prompt, max_new_tokens=32, **sampler_kwargs):
        if sampler_kwargs is None:
            sampler_kwargs = {
                'temperature': 0.01,
                'top_p': 0.1,
                'top_k': 1,
        }
        response = self.model.generate(
            prompt,
            device=self._device,
            output_len=max_new_tokens,
            **sampler_kwargs,
        )
        return response

    def _parse_keyword(self, response: str):
        match = re.search(r"(?<=\*\*)([^*]+)(?=\*\*)", response)
        if match is None:
            keyword = ''
        else:
            keyword = match.group().lower()
        return keyword

    def _parse_response(self, response: str, obs: dict):
        raise NotImplementedError


def interleave_unequal(x, y):
    return [
        item for pair in itertools.zip_longest(x, y) for item in pair if item is not None
    ]


class GemmaQuestionerAgent(GemmaAgent):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def _start_session(self, obs):
        self.formatter.reset()
        self.formatter.user("Let's play 20 Questions. You are playing the role of the Questioner.")
        turns = interleave_unequal(obs.questions, obs.answers)
        self.formatter.apply_turns(turns, start_agent='model')

        if obs.turnType == 'ask':
            self.formatter.user(
                "Ask one short strategic yes-or-no question that helps narrow down the hidden keyword. "
                "Use the previous questions and answers as context. "
                "Avoid random, unrelated, or repeated questions. "
                "The question must be directly useful for identifying a person, place, or thing."
            )

        elif obs.turnType == 'guess':
            self.formatter.user(
                "Based on the previous questions and answers, make your best logical guess for the keyword. "
                "Return only the guessed keyword surrounded by double asterisks, for example **Tanzania**."
            )

        self.formatter.start_model_turn()

    def _parse_response(self, response: str, obs: dict):
        if obs.turnType == 'ask':
            match = re.search(".+?\?", response.replace('*', ''))

            if match is None:
                question = "Is it a place?"
            else:
                question = match.group().strip()

                bad_prefixes = [
                    "question:",
                    "sure, here's your question:",
                    "okay, here's your question:",
                    "the answer is:",
                    "yes, the answer is:",
                    "here's your question:"
                ]

                for prefix in bad_prefixes:
                    if question.lower().startswith(prefix):
                        question = question[len(prefix):].strip()

                if not question.endswith("?"):
                    question += "?"

            return question

        elif obs.turnType == 'guess':
            guess = self._parse_keyword(response)

            if guess == "":
                # fallback if Gemma does not follow **keyword** format
                cleaned = response.lower()
                cleaned = cleaned.replace("guess:", "").replace("the keyword is", "").strip()
                guess = cleaned.split(".")[0].strip()

            return guess

        else:
            raise ValueError("Unknown turn type:", obs.turnType)

class GemmaAnswererAgent(GemmaAgent):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def _start_session(self, obs):
        self.formatter.reset()
        self.formatter.user(f"Let's play 20 Questions. You are playing the role of the Answerer. The keyword is {obs.keyword} in the category {obs.category}.")
        turns = interleave_unequal(obs.questions, obs.answers)
        self.formatter.apply_turns(turns, start_agent='user')
        self.formatter.user(f"The question is about the keyword {obs.keyword} in the category {obs.category}. Give yes-or-no answer and surround your answer with double asterisks, like **yes** or **no**.")
        self.formatter.start_model_turn()

    def _parse_response(self, response: str, obs: dict):
        answer = self._parse_keyword(response)
        return 'yes' if 'yes' in answer else 'no'


# Agent Creation
system_prompt = "You are an AI assistant designed to play the 20 Questions game. In this game, the Answerer thinks of a keyword and responds to yes-or-no questions by the Questioner. The keyword is a specific person, place, or thing."

few_shot_examples = [
    "Is it a place?", "**yes**",
    "Is it a country?", "**yes**",
    "Is it located in Africa?", "**yes**",
    "Is it known for wildlife or safaris?", "**yes**",
    "**Tanzania**", "Correct!"
]

# **IMPORTANT:** Define agent as a global so you only have to load
# the agent you need. Loading both will likely lead to OOM.
agent = None


def get_agent(name: str):
    global agent
    
    if agent is None and name == 'questioner':
        agent = GemmaQuestionerAgent(
            device='cuda:0',
            system_prompt=system_prompt,
            few_shot_examples=few_shot_examples,
        )
    elif agent is None and name == 'answerer':
        agent = GemmaAnswererAgent(
            device='cuda:0',
            system_prompt=system_prompt,
            few_shot_examples=few_shot_examples,
        )
    assert agent is not None, "Agent not initialized."

    return agent


def agent_fn(obs, cfg):
    if obs.turnType == "ask":
        response = get_agent('questioner')(obs)
    elif obs.turnType == "guess":
        response = get_agent('questioner')(obs)
    elif obs.turnType == "answer":
        response = get_agent('answerer')(obs)
    if response is None or len(response) <= 1:
        return "yes"
    else:
        return response

Writing submission/main.py


In [3]:
import sys
sys.path.append("/kaggle/working/submission")

if "main" in sys.modules:
    del sys.modules["main"]

from main import agent_fn

In [4]:
%%bash
cd /kaggle/working
tar -czf submission.tar.gz submission
ls -lh submission.tar.gz

-rw-r--r-- 1 root root 1.5M May  6 03:03 submission.tar.gz


In [5]:
import sys
sys.path.append("/kaggle/working/submission")

from main import agent_fn

class Obs:
    def __init__(self, turnType, questions=None, answers=None, keyword="tanzania", category="place"):
        self.turnType = turnType
        self.questions = questions or []
        self.answers = answers or []
        self.keyword = keyword
        self.category = category

def simple_answer(question, keyword="tanzania"):
    q = question.lower()

    if "person" in q:
        return "no"
    if "place" in q:
        return "yes"
    if "country" in q:
        return "yes"
    if "africa" in q:
        return "yes"
    if "tanzania" in q:
        return "yes"

    return "no"

questions = []
answers = []
keyword = "tanzania"
category = "place"

print("Initializing Gemma questioner baseline simulation...\n")

for i in range(1, 6):
    obs_ask = Obs("ask", questions, answers, keyword, category)
    question = agent_fn(obs_ask, None)
    questions.append(question)

    answer = simple_answer(question, keyword)
    answers.append(answer)

    obs_guess = Obs("guess", questions, answers, keyword, category)
    guess = agent_fn(obs_guess, None)

    print(f"Question {i}: {question}")
    print(f"Answer: {answer}")
    print(f"Guess: {guess}\n")

Initializing Gemma questioner baseline simulation...

Initializing model


/opt/conda/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


response='Is the keyword a city or a town?'
response='the answer is:'
Question 1: Is the keyword a city or a town?
Answer: no
Guess: the answer is:

response='Are the majority of the inhabitants of the place known to be tall or short?'
response='tanzania'
Question 2: Are the majority of the inhabitants of the place known to be tall or short?
Answer: yes
Guess: tanzania

response='Are the historical buildings in the place mostly made of stone or wooden materials?'
response='tanzania'
Question 3: Are the historical buildings in the place mostly made of stone or wooden materials?
Answer: yes
Guess: tanzania

response='Are the streets of the place mostly straight or winding?'
response='tanzania'
Question 4: Are the streets of the place mostly straight or winding?
Answer: yes
Guess: tanzania

response='Are the rivers that flow through the place mostly perennial or seasonal?'
response='tanzania'
Question 5: Are the rivers that flow through the place mostly perennial or seasonal?
Answer: yes
